## Calling API from OpneRouter


# 📞 Your First LLM API Call

Now that we understand what an LLM API is, it's time to make our **first API call**.

An API call is simply a request sent from our Python application to a language model. The request contains our prompt along with some configuration, such as which model to use and how creative the response should be. The language model processes the request and returns a generated response.

This process is called **Inference**. During inference, the model is **not learning or updating its parameters**. Instead, it uses the knowledge it gained during training to predict the next token based on the input prompt.

When we execute the code below, several things happen behind the scenes:

1. Our Python application creates a request.
2. The request is sent to the OpenRouter API.
3. OpenRouter forwards it to the selected language model.
4. The model generates a response token by token.
5. The generated response is returned as a JSON object.
6. Our Python program extracts the response and displays it.

In addition to the generated answer, the API also returns useful metadata such as:

- The model used
- Prompt tokens
- Completion tokens
- Total token usage

Understanding this response is important because token usage directly affects API cost and performance in larger AI applications.

In [1]:
!pip install openai requests -q

In [4]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

### Your first API call

This is **inference**. You send tokens, get tokens back.  the API just wraps it with HTTP.

In [6]:
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{"role": "user", "content": "What is 2 + 2? Answer in one word."}],
    temperature=0.0,
    max_tokens=50,
)

print("Response:", response.choices[0].message.content)
print(f"Tokens — in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

Response: Four
Tokens — in: 26, out: 2
Model used: google/gemma-4-31b-it-20260402:free


### Temperature — quick demo

You know why this works: `softmax(logits / T)`
- T=0: deterministic
- T=1: variety
- T=2: chaos

In [7]:
prompt = "Write a one-sentence story about a robot."

for temp in [0.0, 1.0, 2.0]:
    results = []

    for _ in range(3):
        r = client.chat.completions.create(
            model=FREE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=60,
        )

        content = r.choices[0].message.content

        if content:
            results.append(content.strip())
        else:
            results.append("[No content returned]")

    print(f"\nTemperature = {temp}:")
    for i, text in enumerate(results):
        print(f"  [{i+1}] {text[:120]}")


Temperature = 0.0:
  [1] A curious robot wandered the forest, discovering a hidden garden where flowers whispered secrets to the wind.
  [2] The obsolete gardening unit, scheduled for recycling at dawn
  [3] The robot spent centuries polishing the ruins of the city, waiting for a creator who would never return to tell it that 

Temperature = 1.0:
  [1] [No content returned]
  [2] We need to output a single sentence story about a robot. No constraints beyond that. Provide a creative one-sentence sto
  [3] [No content returned]

Temperature = 2.0:
  [1] The user wants a one-sentence story about a robot.
It needs to be evocative but concise.
Themes could include: vsan sobा
  [2] After centuries of solitude on a dead planet, the robot spent its final spark of energy planting a single, synthetic see
  [3] [No content returned]


### The model has NO memory

Multi-turn: the **entire conversation** is re-sent every time. "Chat memory" is your application re-sending history.

In [12]:
conversation = [
    {"role": "system", "content": "You are a math tutor. Be concise."},
    {"role": "user", "content": "What is a derivative?"},
]

r1 = client.chat.completions.create(
    model=FREE_MODEL,
    messages=conversation,
    max_tokens=150
)

turn1 = r1.choices[0].message.content or "[No content returned]"
print("Turn 1:", turn1[:200])
print(f"  Prompt tokens: {r1.usage.prompt_tokens}")
print(f"Model used: {response.model}")

conversation.append({"role": "assistant", "content": turn1})
conversation.append({"role": "user", "content": "Give me an example with x²."})

r2 = client.chat.completions.create(
    model=FREE_MODEL,
    messages=conversation,
    max_tokens=150
)

turn2 = r2.choices[0].message.content or "[No content returned]"
print(f"\nTurn 2: {turn2[:200]}")
print(f"  Prompt tokens: {r2.usage.prompt_tokens}")
print(f"Model used: {response.model}")
print(f"\n→ Prompt tokens grew from {r1.usage.prompt_tokens} to {r2.usage.prompt_tokens}")

Turn 1: [No content returned]
  Prompt tokens: 37
Model used: google/gemma-4-31b-it-20260402:free

Turn 2: [No content returned]
  Prompt tokens: 47
Model used: google/gemma-4-31b-it-20260402:free

→ Prompt tokens grew from 37 to 47


## 📝 Summary

In this notebook, we made our first LLM API call using OpenRouter and understood how an application communicates with a language model. We learned how to send prompts, receive responses, inspect token usage, and identify the model used for inference. We also explored the **temperature** parameter and observed how it affects the creativity and consistency of the model's output.

With this foundation, we are now ready to move beyond simple conversations and start building AI agents that can use external tools and solve more complex tasks.